<a href="https://colab.research.google.com/github/cksleigen/lg-aimers-demand-forecasting/blob/chanhee/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install koreanize-matplotlib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 42.5 MB/s eta 0:00:00


In [14]:
from google.colab import drive
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
csv_path = '/content/drive/MyDrive/data/train/train.csv'

df = pd.read_csv(csv_path)
df.head()

,영업일자,영업장명_메뉴명,매출수량
0,2023.1.1,느티나무 셀프BBQ_1인 수저세트,0
1,2023.1.2,느티나무 셀프BBQ_1인 수저세트,0
2,2023.1.3,느티나무 셀프BBQ_1인 수저세트,0
3,2023.1.4,느티나무 셀프BBQ_1인 수저세트,0
4,2023.1.5,느티나무 셀프BBQ_1인 수저세트,0


In [10]:
df[df['매출수량'] < 0]

,영업일자,영업장명_메뉴명,매출수량
1837,2023.8.30,"느티나무 셀프BBQ_대여료 60,000원",-1
3453,2023.9.19,느티나무 셀프BBQ_스프라이트 (단체),-18
55068,2023.9.30,미라시아_브런치 4인 패키지,-1
56134,2023.10.2,미라시아_브런치(대인) 주중,-5
64507,2023.5.16,연회장_Cass Beer,-18
64849,2024.4.22,연회장_Cass Beer,-26
65585,2023.5.30,연회장_Conference L2,-1
65693,2023.9.15,연회장_Conference L2,-1
78401,2023.7.17,카페테리아_단체식 18000(신),-80
89446,2023.3.12,포레스트릿_꼬치어묵,-3


In [17]:
df['영업일자'][0]

'2023.1.1'

In [20]:
df[df['영업일자'] == '2023.8.30']

,영업일자,영업장명_메뉴명,매출수량
241,2023.8.30,느티나무 셀프BBQ_1인 수저세트,0
773,2023.8.30,느티나무 셀프BBQ_BBQ55(단체),118
1305,2023.8.30,"느티나무 셀프BBQ_대여료 30,000원",0
1837,2023.8.30,"느티나무 셀프BBQ_대여료 60,000원",-1
2369,2023.8.30,"느티나무 셀프BBQ_대여료 90,000원",0
...,...,...,...
100257,2023.8.30,화담숲카페_메밀미숫가루,3
100789,2023.8.30,화담숲카페_아메리카노 HOT,7
101321,2023.8.30,화담숲카페_아메리카노 ICE,4
101853,2023.8.30,화담숲카페_카페라떼 ICE,0


# 공휴일 확인

In [ ]:
import holidays

In [ ]:
kr_2024_holidays = holidays.CountryHoliday('KR', years=2024)
kr_2023_holidays = holidays.CountryHoliday('KR', years=2023)

In [ ]:
for date, name in sorted(kr_2024_holidays.items()):
    print(date, name)

2024-01-01 New Year's Day
2024-02-09 The day preceding Korean New Year
2024-02-10 Korean New Year
2024-02-11 The second day of Korean New Year
2024-02-12 Alternative holiday for Korean New Year
2024-03-01 Independence Movement Day
2024-04-10 National Assembly Election Day
2024-05-05 Children's Day
2024-05-06 Alternative holiday for Children's Day
2024-05-15 Buddha's Birthday
2024-06-06 Memorial Day
2024-08-15 Liberation Day
2024-09-16 The day preceding Chuseok
2024-09-17 Chuseok
2024-09-18 The second day of Chuseok
2024-10-01 Armed Forces Day
2024-10-03 National Foundation Day
2024-10-09 Hangul Day
2024-12-25 Christmas Day


In [ ]:
for date, name in sorted(kr_2023_holidays.items()):
    print(date, name)

2023-01-01 New Year's Day
2023-01-21 The day preceding Korean New Year
2023-01-22 Korean New Year
2023-01-23 The second day of Korean New Year
2023-01-24 Alternative holiday for Korean New Year
2023-03-01 Independence Movement Day
2023-05-05 Children's Day
2023-05-27 Buddha's Birthday
2023-05-29 Alternative holiday for Buddha's Birthday
2023-06-06 Memorial Day
2023-08-15 Liberation Day
2023-09-28 The day preceding Chuseok
2023-09-29 Chuseok
2023-09-30 The second day of Chuseok
2023-10-02 Temporary Public Holiday
2023-10-03 National Foundation Day
2023-10-09 Hangul Day
2023-12-25 Christmas Day


In [ ]:

# 컬럼명 확인 및 정리
print("원본 데이터 컬럼:", df.columns.tolist())
print("데이터 샘플:")
print(df.head())

# 영업일자를 datetime으로 변환
df['영업일자'] = pd.to_datetime(df['영업일자'])

# 느티나무 셀프BBQ 매장 데이터만 필터링
bbq_data = df[df['영업장명_메뉴명'].str.contains('느티나무 셀프BBQ', na=False)].copy()

print(f"\n느티나무 셀프BBQ 관련 데이터 수: {len(bbq_data)}")
print("느티나무 셀프BBQ 메뉴 목록:")
print(bbq_data['영업장명_메뉴명'].unique())

# 공휴일 목록 정의
holidays_2023 = [
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24',
    '2023-03-01', '2023-05-05', '2023-05-27', '2023-05-29', '2023-06-06',
    '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-02',
    '2023-10-03', '2023-10-09', '2023-12-25'
]

holidays_2024 = [
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12',
    '2024-03-01', '2024-04-10', '2024-05-05', '2024-05-06', '2024-05-15',
    '2024-06-06'
]

all_holidays = holidays_2023 + holidays_2024
holiday_dates = pd.to_datetime(all_holidays)

# 날짜별로 매출 수량 합계 계산
daily_sales = bbq_data.groupby('영업일자')['매출수량'].sum().reset_index()

# 공휴일 여부 컬럼 추가
daily_sales['is_holiday'] = daily_sales['영업일자'].isin(holiday_dates)

# 월 정보 추가
daily_sales['month'] = daily_sales['영업일자'].dt.month

# 학기/방학 시즌 분류 함수
def classify_season(month):
    if month in [3, 4, 5, 6, 9, 10, 11, 12]:
        return '학기중'
    else:  # 1, 2, 7, 8월
        return '방학'

daily_sales['season'] = daily_sales['month'].apply(classify_season)

# 공휴일/평일 라벨 생성
daily_sales['holiday_label'] = daily_sales['is_holiday'].map({True: '공휴일', False: '평일'})

print(f"\n전체 분석 기간: {daily_sales['영업일자'].min()} ~ {daily_sales['영업일자'].max()}")
print(f"총 영업일 수: {len(daily_sales)}")
print(f"공휴일 수: {daily_sales['is_holiday'].sum()}")
print(f"평일 수: {(~daily_sales['is_holiday']).sum()}")

# 시즌별 데이터 분포 확인
print("\n시즌별 데이터 분포:")
season_distribution = daily_sales.groupby(['season', 'holiday_label']).agg({
    '매출수량': ['count', 'mean', 'std', 'sum']
}).round(2)
print(season_distribution)

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. 전체 기간 공휴일 vs 평일 매출 비교
ax1 = axes[0, 0]
holiday_comparison = daily_sales.groupby('holiday_label')['매출수량'].agg(['mean', 'std']).reset_index()
bars1 = ax1.bar(holiday_comparison['holiday_label'], holiday_comparison['mean'],
                yerr=holiday_comparison['std'], capsize=5, alpha=0.7, color=['skyblue', 'lightcoral'])
ax1.set_title('전체 기간: 공휴일 vs 평일 평균 매출', fontsize=12, pad=20)
ax1.set_ylabel('평균 매출수량')
ax1.grid(True, alpha=0.3)

# 막대 위에 값 표시
for bar, mean_val in zip(bars1, holiday_comparison['mean']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{mean_val:.1f}', ha='center', va='bottom')

# 2. 학기중 시즌 공휴일 vs 평일
ax2 = axes[0, 1]
semester_data = daily_sales[daily_sales['season'] == '학기중']
if len(semester_data) > 0:
    semester_comparison = semester_data.groupby('holiday_label')['매출수량'].agg(['mean', 'std']).reset_index()
    bars2 = ax2.bar(semester_comparison['holiday_label'], semester_comparison['mean'],
                    yerr=semester_comparison['std'], capsize=5, alpha=0.7, color=['lightgreen', 'orange'])
    ax2.set_title('학기중 시즌: 공휴일 vs 평일 평균 매출', fontsize=12, pad=20)
    ax2.set_ylabel('평균 매출수량')
    ax2.grid(True, alpha=0.3)

    for bar, mean_val in zip(bars2, semester_comparison['mean']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{mean_val:.1f}', ha='center', va='bottom')
else:
    ax2.text(0.5, 0.5, '학기중 데이터 없음', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('학기중 시즌: 데이터 없음')

# 3. 방학 시즌 공휴일 vs 평일
ax3 = axes[1, 0]
vacation_data = daily_sales[daily_sales['season'] == '방학']
if len(vacation_data) > 0:
    vacation_comparison = vacation_data.groupby('holiday_label')['매출수량'].agg(['mean', 'std']).reset_index()
    bars3 = ax3.bar(vacation_comparison['holiday_label'], vacation_comparison['mean'],
                    yerr=vacation_comparison['std'], capsize=5, alpha=0.7, color=['purple', 'gold'])
    ax3.set_title('방학 시즌: 공휴일 vs 평일 평균 매출', fontsize=12, pad=20)
    ax3.set_ylabel('평균 매출수량')
    ax3.grid(True, alpha=0.3)

    for bar, mean_val in zip(bars3, vacation_comparison['mean']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{mean_val:.1f}', ha='center', va='bottom')
else:
    ax3.text(0.5, 0.5, '방학 데이터 없음', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('방학 시즌: 데이터 없음')

# 4. 박스플롯으로 분포 비교
ax4 = axes[1, 1]
if len(daily_sales) > 0:
    # 시즌과 공휴일 조합 생성
    daily_sales['season_holiday'] = daily_sales['season'] + '_' + daily_sales['holiday_label']

    # 각 조합별로 데이터가 있는지 확인
    combinations = daily_sales['season_holiday'].unique()
    valid_combinations = []
    for combo in combinations:
        if len(daily_sales[daily_sales['season_holiday'] == combo]) > 0:
            valid_combinations.append(combo)

    if valid_combinations:
        box_data = daily_sales[daily_sales['season_holiday'].isin(valid_combinations)]
        sns.boxplot(data=box_data, x='season_holiday', y='매출수량', ax=ax4)
        ax4.set_title('시즌별 공휴일/평일 매출 분포', fontsize=12, pad=20)
        ax4.set_xlabel('시즌_공휴일여부')
        ax4.set_ylabel('매출수량')
        ax4.tick_params(axis='x', rotation=45)
    else:
        ax4.text(0.5, 0.5, '분석 가능한 데이터 없음', ha='center', va='center', transform=ax4.transAxes)

plt.tight_layout()
plt.show()

# 통계적 분석 결과 출력
print("\n=== 상세 분석 결과 ===")

# 전체 통계
print("\n1. 전체 기간 분석:")
overall_stats = daily_sales.groupby('holiday_label')['매출수량'].describe()
print(overall_stats)

# 학기중 통계
print("\n2. 학기중 시즌 분석:")
if len(daily_sales[daily_sales['season'] == '학기중']) > 0:
    semester_stats = daily_sales[daily_sales['season'] == '학기중'].groupby('holiday_label')['매출수량'].describe()
    print(semester_stats)
else:
    print("학기중 데이터가 충분하지 않습니다.")

# 방학 통계
print("\n3. 방학 시즌 분석:")
if len(daily_sales[daily_sales['season'] == '방학']) > 0:
    vacation_stats = daily_sales[daily_sales['season'] == '방학'].groupby('holiday_label')['매출수량'].describe()
    print(vacation_stats)
else:
    print("방학 데이터가 충분하지 않습니다.")

# 월별 매출 패턴
print("\n4. 월별 매출 패턴:")
monthly_pattern = daily_sales.groupby(['month', 'holiday_label'])['매출수량'].mean().unstack(fill_value=0)
print(monthly_pattern.round(2))

# 결론 도출
print("\n=== 분석 결론 ===")
total_holiday_mean = daily_sales[daily_sales['is_holiday']]['매출수량'].mean()
total_weekday_mean = daily_sales[~daily_sales['is_holiday']]['매출수량'].mean()

print(f"전체 기간 평균 매출:")
print(f"- 공휴일: {total_holiday_mean:.2f}")
print(f"- 평일: {total_weekday_mean:.2f}")
print(f"- 차이: {total_holiday_mean - total_weekday_mean:.2f} ({'공휴일이 높음' if total_holiday_mean > total_weekday_mean else '평일이 높음'})")

# 시즌별 비교
for season in ['학기중', '방학']:
    season_data = daily_sales[daily_sales['season'] == season]
    if len(season_data) > 0:
        holiday_mean = season_data[season_data['is_holiday']]['매출수량'].mean()
        weekday_mean = season_data[~season_data['is_holiday']]['매출수량'].mean()

        print(f"\n{season} 시즌 평균 매출:")
        print(f"- 공휴일: {holiday_mean:.2f}")
        print(f"- 평일: {weekday_mean:.2f}")
        print(f"- 차이: {holiday_mean - weekday_mean:.2f} ({'공휴일이 높음' if holiday_mean > weekday_mean else '평일이 높음'})")